# Introduction to Pandas

**pandas** is a Python package providing fast, flexible, and expressive data structures designed to work with *relational* or *labeled* data both. It is a fundamental high-level building block for executing practical, real world data analysis in Python. 

pandas is well suited for:

- Tabular data with heterogeneously-typed columns, as in an SQL table or Excel spreadsheet
- Ordered and unordered (not necessarily fixed-frequency) time series data.
- Arbitrary matrix data (homogeneously typed or heterogeneous) with row and column labels
- Any other form of observational / statistical data sets. The data actually need not be labeled at all to be placed into a pandas data structure


Key features:
    
- Easy handling of **missing data**
- **Size mutability**: columns can be inserted and deleted from DataFrame and higher dimensional objects
- Automatic and explicit **data alignment**: objects can be explicitly aligned to a set of labels, or the data can be aligned automatically
- Powerful, flexible **group by functionality** to perform split-apply-combine operations on data sets
- Intelligent label-based **slicing, fancy indexing, and subsetting** of large data sets
- Intuitive **merging and joining** data sets
- Flexible **reshaping and pivoting** of data sets
- **Hierarchical labeling** of axes
- Robust **IO tools** for loading data from flat files, Excel files, databases, and HDF5
- **Time series functionality**: date range generation and frequency conversion, moving window statistics, moving window linear regressions, date shifting and lagging, etc.

## Installing and Using Pandas

Installation of Pandas on your system requires NumPy to be installed, and if building the library from source, requires the appropriate tools to compile the C and Cython sources on which Pandas is built.
Details on this installation can be found in the [Pandas documentation](http://pandas.pydata.org/).

Once Pandas is installed, you can import it and check the version:

In [ ]:
import pandas
pandas.__version__

# Introducing Pandas Objects

At the very basic level, Pandas objects can be thought as enhanced versions of NumPy structured arrays in which the rows and columns are identified with labels rather than simple integer indices.
As we will see during the course of this chapter, Pandas provide a host of useful tools, methods, and functionality on top of the basic data structures, but nearly everything that follows will require an understanding of what these structures are.
Thus, before we go any further, let's introduce these three fundamental Pandas data structures: the ``Series``, ``DataFrame``, and ``Index``.

We will start our code sessions with the standard NumPy and Pandas imports:

In [2]:
import numpy as np
import pandas as pd

### Series

A **Series** is a single vector of data (like a NumPy array) with an *index* that labels each element in the vector.

In [ ]:
counts = pd.Series([632, 1638, 569, 115])
counts

If an index is not specified, a default sequence of integers is assigned as the index. A NumPy array comprises the values of the `Series`, while the index is a pandas `Index` object.

In [ ]:
counts.values

In [ ]:
counts.index

We can assign meaningful labels to the index, if they are available:

In [ ]:
bacteria = pd.Series([632, 1638, 569, 115], 
    index=['Firmicutes', 'Proteobacteria', 'Actinobacteria', 'Bacteroidetes'])

bacteria

These labels can be used to refer to the values in the `Series`.

In [ ]:
bacteria['Actinobacteria']

Notice that the indexing operation preserve the association between the values and the corresponding indices.

We can still use positional indexing if we wish.

In [ ]:
bacteria[0]

We can give both the array of values and the index meaningful labels themselves:

In [ ]:
bacteria.name = 'counts'
bacteria.index.name = 'phylum'
bacteria

NumPy's math functions and other operations can be applied to Series without losing the data structure.

In [ ]:
np.log(bacteria)

We can also filter according to the values in the `Series`:

In [ ]:
bacteria[bacteria>1000]

A `Series` can be thought of as an ordered key-value store. In fact, we can create one from a `dict`:

In [ ]:
bacteria_dict = {'Firmicutes': 632, 'Proteobacteria': 1500, 'Actinobacteria': 569, 'Bacteroidetes': 115}
print(bacteria_dict)
pd.Series(bacteria_dict)

### ``Series`` as generalized NumPy array

From what we've seen so far, it may look like the ``Series`` object is basically interchangeable with a one-dimensional NumPy array.
The essential difference is the presence of the index: while the Numpy Array has an *implicitly defined* integer index used to access the values, the Pandas ``Series`` has an *explicitly defined* index associated with the values.

This explicit index definition gives the ``Series`` object additional capabilities.

### Series as specialized dictionary

In this way, you can think of a Pandas ``Series`` a bit like a specialization of  Python dictionary.
A dictionary is a structure that maps arbitrary keys to set of arbitrary values, and ``Series`` is a structure which maps typed keys to a set of typed values.
This typing is important: just as the type-specific compiled code behind a NumPy array makes it more efficient than Python list for certain operations, the type information of a Pandas ``Series`` makes it much more efficient than Python dictionaries for certain operations.

The ``Series``-as-dictionary analogy can be made even more clear by constructing a ``Series`` object directly from Python dictionary:

## DataFrame: bi-dimensional Series with two (or more) indices

A DataFrame represents a tabular, spreadsheet-like data structure containing an ordered collection of columns, each of which can be a different value type (numeric, string, boolean, etc.). 

The DataFrame has both row and column index; it can be thought of as a dict of Series (all sharing the same index).

DataFrame's can be created in different ways. One of them is from dict's.

In [ ]:
data = {"Province": ["FL", "FL", "NH", "NH", "ZH"],
        "Year": [2013, 2014, 2013, 2014, 2014],
        "Literacy": [0.2, 0.1, 0.5, 0.3, 0.5]}
print(data)
data = pd.DataFrame(data)
data

To change the order of the columns:

In [ ]:
df = pd.DataFrame(data, columns=["Year", "Province" ,"Literacy"])
df

An `index` can be passed (as with Series), and passing column names not existing, will result in missing data.

Assigning values to new columns is easy

In [ ]:
df['nonsense'] = df.Year / df.Literacy
df

In [ ]:
df['Serie_aligned'] = pd.Series(range(5), index=[0,1,2, 3, 4])
df

Passing a dicts where the values are dicts is also possible

In [ ]:
df.to_dict()

In [ ]:
pd.DataFrame(df.to_dict())

# Operating on Data in Pandas

One of the essential pieces of NumPy is the ability to perform quick element-wise operations, both with basic arithmetic (addition, subtraction, multiplication, etc.) and with more sophisticated operations (trigonometric functions, exponential and logarithmic functions, etc.).
Pandas inherits much of this functionality from NumPy, and the ufuncs are key to this.

Pandas includes a couple useful twists, however: for unary operations like negation and trigonometric functions, these ufuncs will *preserve index and column labels* in the output, and for binary operations such as addition and multiplication, Pandas will automatically *align indices* when passing the objects to the ufunc.
This means that keeping the context of data and combining data from different sources–both potentially error-prone tasks with raw NumPy arrays–become essentially foolproof with Pandas.
We will additionally see the well-defined operations between one-dimensional ``Series`` structures and two-dimensional ``DataFrame`` structures.

## Ufuncs: Index Preservation

Because Pandas is designed to work with NumPy, any NumPy ufunc will work on Pandas ``Series`` and ``DataFrame`` objects.
Let's start by defining a simple ``Series`` and ``DataFrame`` on which to demonstrate this:

In [6]:
ser = pd.Series(np.random.randint(0, 10, 4))
ser

0    3
1    3
2    8
3    3
dtype: int32

In [10]:
rng = np.random.RandomState(10)
ser = pd.Series(rng.randint(0, 10, 4))
ser

0    9
1    4
2    0
3    1
dtype: int32

In [17]:
dfr = pd.DataFrame(rng.randint(0, 10, (8, 4)),
                  columns=['A', 'B', 'C', 'D'])
dfr

,A,B,C,D
0,5,2,3,4
1,5,7,6,8
2,0,4,6,3
3,5,9,3,8
4,3,0,9,6
5,2,3,9,1
6,4,0,5,0
7,8,1,3,6


If we apply a NumPy ufunc on either of these objects, the result will be another Pandas object *with the indices preserved:*

In [18]:
np.exp(ser)

0    8103.083928
1      54.598150
2       1.000000
3       2.718282
dtype: float64

Or, for a slightly more complex calculation:

In [19]:
np.sin(dfr * np.pi / 4)

,A,B,C,D
0,-7.071068e-01,1.000000e+00,0.707107,1.224647e-16
1,-7.071068e-01,-7.071068e-01,-1.000000,-2.449294e-16
2,0.000000e+00,1.224647e-16,-1.000000,7.071068e-01
3,-7.071068e-01,7.071068e-01,0.707107,-2.449294e-16
4,7.071068e-01,0.000000e+00,0.707107,-1.000000e+00
5,1.000000e+00,7.071068e-01,0.707107,7.071068e-01
6,1.224647e-16,0.000000e+00,-0.707107,0.000000e+00
7,-2.449294e-16,7.071068e-01,0.707107,-1.000000e+00


## Universal Functions: Index Alignment

For binary operations on two ``Series`` or ``DataFrame`` objects, Pandas will align indices in the process of performing the operation.
This is very convenient when working with incomplete data, as we'll see in some of the examples that follow.

### Index alignment in Series

As an example, suppose we are combining two different data sources, and find only the top three US states by *area* and the top three US states by *population*:

In [26]:
area = pd.Series({'Alaska': 1723337, 'Texas': 695662,
                  'California': 423967}, name='area')
population = pd.Series({'California': 38332521, 'Texas': 26448193,
                        'New York': 19651127}, name='population')
print(area)
population


Alaska        1723337
Texas          695662
California     423967
Name: area, dtype: int64


California    38332521
Texas         26448193
New York      19651127
Name: population, dtype: int64

Let's see what happens when we divide these to compute the population density:

In [21]:
population / area

Alaska              NaN
California    90.413926
New York            NaN
Texas         38.018740
dtype: float64

The resulting array contains the *union* of indices of the two input arrays, which could be determined using standard Python set arithmetic on these indices:

In [27]:
area.index.union(population.index)

Index(['Alaska', 'California', 'New York', 'Texas'], dtype='object')

In [23]:
A = pd.Series([2, 4, 6], index=[0, 1, 2])
B = pd.Series([1, 3, 5], index=[1, 2, 3])
print(A)
print(B)
B
A + B

0    2
1    4
2    6
dtype: int64
1    1
2    3
3    5
dtype: int64


0    NaN
1    5.0
2    9.0
3    NaN
dtype: float64

If using NaN values is not the desired behavior, the fill value can be modified using appropriate object methods in place of the operators.
For example, calling ``A.add(B)`` is equivalent to calling ``A + B``, but allows optional explicit specification of the fill value for any elements in ``A`` or ``B`` that might be missing:

In [24]:
A.add(B, fill_value=0)

0    2.0
1    5.0
2    9.0
3    5.0
dtype: float64

The following table lists Python operators and their equivalent Pandas object methods:

| Python Operator | Pandas Method(s)                      |
|-----------------|---------------------------------------|
| ``+``           | ``add()``                             |
| ``-``           | ``sub()``, ``subtract()``             |
| ``*``           | ``mul()``, ``multiply()``             |
| ``/``           | ``truediv()``, ``div()``, ``divide()``|
| ``//``          | ``floordiv()``                        |
| ``%``           | ``mod()``                             |
| ``**``          | ``pow()``                             |


Groupby